# 00 - End-to-end workflow

This notebook runs the whole system by calling the functions in `scripts/` (the same steps as
`python main.py`): download → transform → predict → simulate → latest trades.

| Step | Module | Output |
|---|---|---|
| 1. Data (incremental) | `scripts/data_repo.py` | `data/raw/*.parquet` |
| 2. Transformations | `scripts/transform.py` | `data/processed/dataset.parquet` |
| 3. Models (load or train) | `scripts/train.py` | `data/models/models.joblib`, `predictions.parquet` |
| 4. Simulation | `scripts/simulate.py` | strategy vs benchmarks |
| 5. Report | `scripts/report.py` | `output/latest_signals.csv`, `output/daily_report.md` |

In [1]:
import sys
sys.path.insert(0, "..")
import time

import pandas as pd

import config as cfg
from scripts.data_repo import DataRepository
from scripts.report import latest_signals, model_health, write_report
from scripts.simulate import benchmark_equity, exact_simulation, performance, risk_free_rate
from scripts.train import TrainModel
from scripts.transform import TransformData

pd.set_option("display.width", 160)
FAST = False   # True: skip the download and the transformations (use the files in data/)

## Step 1 - Data
Incremental update: only the last days are downloaded; tickers whose past prices changed
(dividend / split adjustments) are downloaded again in full.

In [2]:
t0 = time.time()
repo = DataRepository()
if FAST:
    repo.load()
else:
    repo.fetch(incremental=True)
    repo.persist()
print(repo.summary(), f"({time.time() - t0:.0f}s)")
repo.universe.groupby("Market").size()

  universe (B3 API + config)...


  stocks: 239 tickers (yfinance)...


  incremental: 1734 recent rows, 0 new tickers, 0 re-adjusted tickers


  market series (yfinance)...


  incremental: 171 recent rows, 0 new tickers, 0 re-adjusted tickers
  macro (Banco Central do Brasil + BCRA)...


stocks: 1,011,291 rows, 239 tickers, 2004-01-01 -> 2026-10-21 | market: (5947, 22) | macro: (8330, 4) (402s)


Market
AN      7
AR     18
BR    148
CL     30
MX     36
dtype: int64

## Step 2 - Transformations

In [3]:
t0 = time.time()
td = TransformData(repo)
if FAST:
    td.load()
else:
    td.transform()
    td.persist()
print(td.df.shape, f"({time.time() - t0:.0f}s)")

  cleaning removed 56,465 bad rows; per-ticker features: (954626, 132) (21s)


  dataset: (937505, 216), 0.72 GB in memory (48s)


(937505, 216) (178s)


## Step 3 - Models: load the saved models and predict (use `tm.select_features().tune().fit_final()` to retrain)

In [4]:
tm = TrainModel(td.df).prepare()
del td
tm.load()
pred = tm.predict()
tm.persist()
pred.groupby("split")["Date"].agg(["min", "max", "size"])

  features: 251 | rows per split: {'train': 331495, 'test': 230919, 'validation': 115887, 'no_target': 15026, 'embargo': 6698, 'live': 4252}


,min,max,size
split,,,
live,2026-09-07,2026-10-21,4252
test,2022-01-03,2026-09-23,230919
validation,2019-01-02,2021-12-01,115887


## Step 4 - Simulation of the selected strategy on the test period

In [5]:
st = cfg.STRATEGY
st

{'prediction': 'pred_xgb',
 'k': 5,
 'threshold': 0.5,
 'stop_loss': 0.1,
 'take_profit': None,
 'size_by_prob': True,
 'regime_filter': False}

In [6]:
test = pred[pred["split"] == "test"]
equity, trades = exact_simulation(test, st["prediction"], k=st["k"], threshold=st["threshold"],
                                  stop_loss=st["stop_loss"], take_profit=st["take_profit"],
                                  size_by_prob=st["size_by_prob"], regime_filter=st["regime_filter"])
bench = benchmark_equity(repo.market, equity.index)
rf = risk_free_rate(repo.market, equity.index)
summary = pd.DataFrame([{"portfolio": "strategy", **performance(equity, rf)}]
                       + [{"portfolio": c, **performance(bench[c], rf)} for c in bench.columns])
summary.round(3)

,portfolio,cagr,annual_vol,sharpe,max_drawdown,calmar,total_return
0,strategy,1.179,0.149,0.885,-0.224,0.800,1.174
1,ILF (Latin America 40),1.161,0.226,0.581,-0.297,0.540,1.019
2,EWZ (Brazil),1.143,0.266,0.472,-0.322,0.442,0.875
3,S&P 500,1.106,0.169,0.429,-0.254,0.416,0.607


## Step 5 - Latest signals (stocks to buy at the next session) and report

In [7]:
signals = latest_signals(pred, st["prediction"], k=st["k"], threshold=st["threshold"])
health = model_health(pred, st["prediction"])
print(write_report(signals, health, summary, st["prediction"]))

# LatAm stock signals - 2026-10-21

Model: `pred_xgb` | horizon: 20 trading days | generated 2026-10-21 20:03

## Buy candidates (beat the country ETF in the next 20 days)

| Ticker       | Market   | Sector           |   Close |   prob | market_regime   |
|:-------------|:---------|:-----------------|--------:|-------:|:----------------|
| FUNO11.MX    | MX       | Real Estate      |   32.96 |  0.608 | bear            |
| KOFUBL.MX    | MX       | Consumer Staples |  220.2  |  0.606 | bear            |
| CBAV3.SA     | BR       | Materials        |   11.22 |  0.606 | bull            |
| VESTA.MX     | MX       | Real Estate      |   67.55 |  0.591 | bear            |
| FIBRAMQ12.MX | MX       | Real Estate      |   46.03 |  0.59  | bear            |

## Model health (last 120 days with known outcome)

- Top-5 precision: 42.8% (base rate 45.8%)
- Average excess return of the top-5 vs the country ETF: -0.93%

## Strategy (test period, exact simulation)

| portfolio              |   cagr